In [1]:
from typing import Dict, List, Optional


class BodaBusiness:

    def __init__(self, name: Optional[str] = None, fare: float = 0.0, dust: float = 0.0, distance: float = 0.0, time: float = 0.0):
        self._fare = int(round(fare))
        self._dust = int(round(dust))
        self._distance = int(round(distance))
        self._time = int(round(time))
        self.name = name if name is not None else "Unknown"
        self._trips: List[Dict] = []
        self._is_available = True
        self._vehicle_condition = "good"
        self._rating: Optional[float] = None

    @property
    def fare(self) -> int:
        return self._fare

    @fare.setter
    def fare(self, value: float) -> None:
        value = int(round(value))
        if value < 0:
            raise ValueError("fare must be non-negative")
        self._fare = value

    @property
    def dust(self) -> int:
        return self._dust

    @dust.setter
    def dust(self, value: float) -> None:
        value = int(round(value))
        if value < 0:
            raise ValueError("dust must be non-negative")
        self._dust = value

    @property
    def distance(self) -> int:
        return self._distance

    @distance.setter
    def distance(self, value: float) -> None:
        value = int(round(value))
        if value < 0:
            raise ValueError("distance must be non-negative")
        self._distance = value

    @property
    def time(self) -> int:
        return self._time

    @time.setter
    def time(self, value: float) -> None:
        value = int(round(value))
        if value < 0:
            raise ValueError("time must be non-negative")
        self._time = value

    def trips(self) -> List[Dict]:
        return list(self._trips)

    def is_available(self) -> bool:
        return self._is_available

    def set_availability(self, available: bool) -> None:
        self._is_available = bool(available)

    def update_vehicle_condition(self, condition: str) -> None:
        self._vehicle_condition = condition

    def add_rating(self, rating: float) -> None:
        rating = float(rating)
        if not 0 <= rating <= 5:
            raise ValueError("rating must be between 0 and 5")
        self._rating = rating

    def calculate_fare(self, base: float = 1.0, per_km: float = 0.5, per_min: float = 0.2, surge: float = 1.0, include_dust_penalty: bool = False) -> int:
        dust_amount = self._dust if include_dust_penalty else 0
        amount = base + self._distance * per_km + self._time * per_min + dust_amount * 0.1
        return int(round(max(amount * float(surge), 0.0)))

    @staticmethod
    def estimate_fare(distance: float, time: float, dust: float = 0.0, base: float = 1.0, per_km: float = 0.5, per_min: float = 0.2, surge: float = 1.0) -> int:
        amount = base + float(distance) * per_km + float(time) * per_min + float(dust) * 0.1
        return int(round(max(amount * float(surge), 0.0)))

    def record_trip(self, fare: float, distance: float, time: float, dust: float = 0.0, weight: float = 0.0, passenger: Optional[str] = None) -> None:
        values = {
            "fare": int(round(fare)),
            "distance": int(round(distance)),
            "time": int(round(time)),
            "dust": int(round(dust)),
            "weight": int(round(weight)),
            "passenger": passenger,
        }
        if any(values[key] < 0 for key in ("fare", "distance", "time", "dust", "weight")):
            raise ValueError("trip values must be non-negative")
        self._trips.append(values)

    def to_dict(self) -> Dict:
        return {
            "name": self.name,
            "fare": self._fare,
            "dust": self._dust,
            "distance": self._distance,
            "time": self._time,
            "total_fare": sum(t["fare"] for t in self._trips),
            "trips": list(self._trips),
            "total_weight": sum(t.get("weight", 0) for t in self._trips),
            "is_available": self._is_available,
            "vehicle_condition": self._vehicle_condition,
            "rating": self._rating,
        }

    @classmethod
    def from_dict(cls, data: Dict) -> "BodaBusiness":
        inst = cls(
            name=data.get("name", "Unknown"),
            fare=data.get("fare", 0.0),
            dust=data.get("dust", 0.0),
            distance=data.get("distance", 0.0),
            time=data.get("time", 0.0),
        )
        inst._trips = list(data.get("trips", []))
        inst._is_available = bool(data.get("is_available", True))
        inst._vehicle_condition = data.get("vehicle_condition", "good")
        inst._rating = data.get("rating")
        return inst

    def detailed_summary(self) -> str:
        lines = [
            f"Name: {self.name}",
            f"Total fare: Sh {sum(t['fare'] for t in self._trips)}",
            f"Trips: {len(self._trips)}",
            f"Total distance: {sum(t['distance'] for t in self._trips)} km",
            f"Total time: {sum(t['time'] for t in self._trips)} min",
            f"Total dust: {sum(t.get('dust', 0) for t in self._trips)}",
            f"Total weight: {sum(t.get('weight', 0) for t in self._trips)} kg",
            "",
            "Trips detail:",
        ]
        for i, trip in enumerate(self._trips, start=1):
            lines.append(
                f" {i}. fare=Sh {trip['fare']}, distance={trip['distance']}km, "
                f"time={trip['time']}min, dust={trip.get('dust', 0)}, "
                f"weight={trip.get('weight', 0)}kg, passenger={trip.get('passenger')}"
            )
        return "\n".join(lines)

    def __repr__(self) -> str:
        return f"BodaBusiness(name={self.name!r}, trips={len(self._trips)})"


bb = BodaBusiness(name="imran")
bb.record_trip(fare=5.5, distance=2.3, time=10, weight=50)
bb.record_trip(fare=7.25, distance=3.8, time=15, dust=1.2, weight=30, passenger="Alice")
print(bb.detailed_summary())

Name: imran
Total fare: Sh 13
Trips: 2
Total distance: 6 km
Total time: 25 min
Total dust: 1
Total weight: 80 kg

Trips detail:
 1. fare=Sh 6, distance=2km, time=10min, dust=0, weight=50kg, passenger=None
 2. fare=Sh 7, distance=4km, time=15min, dust=1, weight=30kg, passenger=Alice
